In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
payrolls = pd.read_csv("mlb_payrolls_2026.csv")
mlb_40man = pd.read_csv("mlb_40man_acquisition.csv")

In [5]:
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup

from data import df as playoff_teams

BR_CODES = {"TBD": "TBR", "KC": "KCR", "WSH": "WSN"}

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/115.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://google.com",
}

player_rows = []

for _, row in playoff_teams.iterrows():
    year = row["Year"]
    team = BR_CODES.get(row["NAME"], row["NAME"])
    url = f"https://www.baseball-reference.com/teams/{team}/{year}.shtml"
    response = requests.get(url, headers=headers)
    response.encoding = "utf-8"

    if response.status_code != 200:
        print(f"Failed to fetch {team} {year}: Status {response.status_code}")
        time.sleep(4)
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    meta = re.sub(r"\s+", " ", soup.find(id="meta").get_text(" ", strip=True))

    record = re.search(r"Record:\s*(\d+)-(\d+),\s*(?:Finished\s*)?(\w+?)(?:\s*place)?\s*in\s*(\w+)", meta)
    wins, losses, place, division = (
        (int(record.group(1)), int(record.group(2)), record.group(3), record.group(4))
        if record else (None, None, None, None)
    )
    post = re.search(r"Postseason:\s*(.*?)\s*Manager:", meta)
    results = re.split(r"\s(?=(?:Won|Lost) )", re.sub(r"\s+\(", " (", post.group(1))) if post else []

    players = {}
    for table_id in ["players_standard_batting", "players_standard_pitching"]:
        table = soup.find("table", id=table_id)
        if not table:
            continue
        for tr in table.find("tbody").find_all("tr"):
            name = tr.find(attrs={"data-stat": "name_display"})
            if name is None or "thead" in (tr.get("class") or []):
                continue
            pos = tr.find(attrs={"data-stat": "team_position"})
            age = tr.find(attrs={"data-stat": "age"})
            players.setdefault(name.get_text(strip=True).rstrip("*#"), {
                "Position": pos.get_text(strip=True) if pos else None,
                "Age": age.get_text(strip=True) if age else None,
            })

    for name, info in players.items():
        player_rows.append({
            "Name": name, **info, "Team": team, "Year": year,
            "Wins": wins, "Losses": losses, "Place": place, "Division": division,
            "Playoff Result": results[0] if results else None,
            "Postseason": "; ".join(results) if results else None,
        })

    print(team, year, wins, results[0] if results else None, len(players))
    time.sleep(4)

team_rosters = pd.DataFrame(player_rows)
team_rosters["Age"] = pd.to_numeric(team_rosters["Age"], errors="coerce")



KCR 2015 95 Won World Series (4-1) over New York Mets 45
TOR 2015 93 Lost AL Championship Series (4-2) to Kansas City Royals 52
TEX 2015 88 Lost AL Division Series (3-2) to Toronto Blue Jays 57
NYY 2015 87 Lost AL Wild Card Game (1-0) to Houston Astros 56
HOU 2015 86 Lost AL Division Series (3-2) to Kansas City Royals 46
NYM 2015 90 Lost World Series (4-1) to Kansas City Royals 49
LAD 2015 92 Lost NL Division Series (3-2) to New York Mets 55
STL 2015 100 Lost NL Division Series (3-1) to Chicago Cubs 46
CHC 2015 97 Lost NL Championship Series (4-0) to New York Mets 50
PIT 2015 98 Lost NL Wild Card Game (1-0) to Chicago Cubs 46
CLE 2016 94 Lost World Series (4-3) to Chicago Cubs 48
TEX 2016 95 Lost AL Division Series (3-0) to Toronto Blue Jays 52
BOS 2016 93 Lost AL Division Series (3-0) to Cleveland Indians 50
TOR 2016 89 Lost AL Championship Series (4-1) to Cleveland Indians 49
BAL 2016 89 Lost AL Wild Card Game (1-0) to Toronto Blue Jays 47
CHC 2016 None Won World Series (4-3) over Cl

In [6]:
import io
import pandas as pd
import requests

SHEET_ID = "12XSXOQpjDJDCJKsA4xC1e_9FlS11aeioZy_p1nqpclg"

SALARY_GIDS = {
    2015: 1369722664,
    2016: 1187664277,
    2017: 1656350490,
    2018: 1775573725,
    2019: 1932759411,
    2020: 285575536,
    2021: 1633405383,
    2022: 1811400237,
    2023: 1677021452,
    2024: 1937251654,
    2025: 387264548,
    2026: 162500738,
}

salary_dfs = {}

for year, gid in SALARY_GIDS.items():
    url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={gid}"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed to fetch {year}: Status {response.status_code}")
        continue

    df = pd.read_csv(io.StringIO(response.text), header=1, usecols=[0, 1, 2, 3])
    df.columns = ["Name", "Pos", "MLS", "Salary"]
    df["Year"] = year
    salary_dfs[year] = df

salaries_all = pd.concat(salary_dfs.values(), ignore_index=True)

salaries_all["Name"] = salaries_all["Name"].astype(str).str.replace("*", "", regex=False).str.strip()
salaries_all["Salary"] = (
    salaries_all["Salary"]
    .astype(str)
    .str.replace(r"[^\d.-]", "", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
)
salaries_all = salaries_all[salaries_all["Name"].str.upper() != "TOTAL"].dropna(subset=["Salary"])
salaries_all["Salary"] = salaries_all["Salary"].round().astype("int64")
salaries_all["MLS"] = pd.to_numeric(salaries_all["MLS"], errors="coerce")

salaries_all.groupby("Year")["Salary"].agg(["count", "max", "sum"])

,count,max,sum
Year,,,
2015,889,32571429,3752129082
2016,871,34571429,3867642785
2017,892,35571429,4099132814
2018,879,35571429,4020474634
2019,875,38333333,3887858407
2020,899,38500000,3987209077
2021,905,37116667,3803830456
2022,974,43333333,4335687674
2023,958,43333333,4714113679


In [1]:
from parse import standardize_names
salaries_all_clean = standardize_names(salaries_all, "Name")

playoff_salary_df = pd.merge(
    team_rosters,
    salaries_all_clean,
    left_on=['Name', 'Year'],
    right_on=['Name', 'Year'],
    how='left'
)

playoff_salary_df.drop(columns=["Playoff Result", "Pos"], inplace=True)


NameError: name 'salaries_all' is not defined

Index(['Name', 'Pos', 'MLS', 'Salary', 'Year'], dtype='object')

518

In [6]:
summary = all_teams.groupby(["team", "year"]).agg(
    Players=("2026", "count"),
    Max2026Sal=("2026", "max"),
    Avg2026Sal=("2026", "mean"),
    Total2026Sal=("2026", "sum"),
)

# Players in 90th percentile of average yearly salary
all_teams["Top10PercSalary"] = all_teams["TotalAvgSalary"] >= all_teams["TotalAvgSalary"].quantile(0.9)

top_10 = all_teams.query("Top10PercSalary == True").groupby(["team", "year"]).agg(
    PlayersInTop10=("Top10PercSalary", "count"),
    Top10AvgSal=("TotalAvgSalary", "mean"),
    Top10TotalSal=("TotalOnContract", "sum")
)

# Players in 90th percentile of average yearly salary
all_teams["Top50PercSalary"] = all_teams["TotalAvgSalary"] >= all_teams["TotalAvgSalary"].quantile(0.5)

top_50 = all_teams.query("Top50PercSalary == True").groupby(["team", "year"]).agg(
    PlayersInTop50=("Top50PercSalary", "count"),
    Top50AvgSal=("TotalAvgSalary", "mean"),
    Top50TotalSal=("TotalOnContract", "sum")
)

In [8]:
acq_counts = (
    all_teams.groupby(["team", "year"])["Acquired"]
    .value_counts(normalize=True) # normalize counts to standardize for roster size
    .unstack(fill_value=0)
)

# provide additional context to non-specified acquisitions
acq_counts.rename(
    columns = lambda c: "Unspecified" if pd.isna(c) or str.strip(c) == "" else c
    )

Acquired,Unspecified,Amateur Draft,Amateur Free Agent,Free Agency,Purchased,Rule 5 Draft,Traded,Waivers
team,,,,,,,,
ATL,0.068182,0.204545,0.045455,0.363636,0.000000,0.000000,0.227273,0.090909
BOS,0.025000,0.175000,0.000000,0.250000,0.000000,0.025000,0.450000,0.075000
CHC,0.137255,0.137255,0.000000,0.333333,0.000000,0.000000,0.313725,0.078431
CHW,0.026316,0.157895,0.000000,0.157895,0.052632,0.026316,0.473684,0.105263
CLE,0.026316,0.315789,0.026316,0.289474,0.026316,0.000000,0.289474,0.026316
HOU,0.000000,0.210526,0.078947,0.236842,0.026316,0.026316,0.315789,0.105263
LAD,0.102041,0.183673,0.000000,0.367347,0.020408,0.000000,0.306122,0.020408
MIL,0.000000,0.200000,0.000000,0.225000,0.050000,0.000000,0.500000,0.025000
NYY,0.166667,0.142857,0.000000,0.333333,0.000000,0.000000,0.309524,0.047619
